# MI vs Magnitude — automated comparison
Runs prune → finetune → sample → metrics for each method, then prints one table.
SSIM (vs the original model, same seed) is the primary metric; FID is optional.

## Setup

In [ ]:
!git clone --branch mipp https://github.com/elliotcanter11/Diff-Pruning.git

In [ ]:
%cd Diff-Pruning/

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -q pytorch-msssim

In [ ]:
!python tools/extract_cifar10_hug.py --output data

In [ ]:
!bash tools/convert_cifar10_ddpm_ema.sh

In [ ]:
# only needed if COMPUTE_FID = True below (mkdir so np.savez has a dir to write into)
!mkdir -p run && python fid_score.py --save-stats data/cifar10_images run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256

## Config + run
SSIM is enough to compare criteria; leave `COMPUTE_FID = False` for fast iteration (samples only ~1k images instead of 10k). Turn it on for a final number.

The table reports, per method: params/MACs after pruning (sanity that greedy hit the ratio), **SSIM** (higher = closer to the original model), and for the MI methods the **overlap vs magnitude** (Jaccard of the pruned set + Spearman) — low overlap = MI is genuinely choosing different filters.

In [ ]:
import os, subprocess, re

RATIO        = 0.3     # pruning ratio (0.5 stresses harder but noisier)
ITERS        = 1000    # finetuning steps (finetune seed is fixed, so runs are comparable)
WORKERS      = 12
SSIM_SAMPLES = 1000    # paired vs the original model; cheap + low-noise
COMPUTE_FID  = False   # SSIM alone is enough to iterate; True for a final FID
FID_SAMPLES  = 10000
DDIM_STEPS   = 100     # 50 ~halves sampling time; fair since all methods match
SAMPLE_BS    = 256     # raise if GPU VRAM is free (check nvidia-smi) -> faster sampling, no downside

# prune_ddpm_cifar10_mi.sh args: ratio  w_output  w_adjacency
METHODS = {
    'magnitude'   : f'bash scripts/prune_ddpm_cifar10.sh {RATIO}',
    'mi_adjacency': f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 0.0 1.0',
    'mi_output'   : f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 1.0 0.0',
    # 'mi_combined' : f'bash scripts/prune_ddpm_cifar10_mi.sh {RATIO} 1.0 1.0',
}

def run(cmd):                         # stream (long steps: finetune/sample)
    print('\n$', cmd, flush=True)
    assert os.system(cmd) == 0, f'FAILED: {cmd}'

def cap(cmd):                         # capture (prune: parse params/macs/overlap)
    print('\n$', cmd, flush=True)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(p.stdout[-1500:])
    if p.returncode != 0:
        print('STDERR', p.stderr[-1500:]); raise SystemExit(f'FAILED: {cmd}')
    return p.stdout

def grab(pat, txt):
    m = re.search(pat, txt); return float(m.group(1)) if m else float('nan')

def metric(cmd, pat, label):         # capture, parse, and SURFACE errors (no silent nan)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    m = re.search(pat, p.stdout)
    if m:
        print(f'{label} = {m.group(1)}')
        return float(m.group(1))
    print(f'!! {label} FAILED (returncode {p.returncode}) -- stderr tail:')
    print((p.stderr or p.stdout)[-800:])
    return float('nan')

def ssim_score():
    return metric('python ddpm_exp/compute_ssim.py '
        '--path run/sample/ddpm_cifar10_pruned/process_0 run/sample/ddpm_cifar10_pretrained/process_0',
        r'ssim:\s*(?:tensor\()?([\-\d.]+)', 'SSIM')

def fid_score():
    return metric('python fid_score.py run/sample/ddpm_cifar10_pruned '
        'run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256',
        r'FID:\s*([\d.]+)', 'FID')

n_samples = FID_SAMPLES if COMPUTE_FID else SSIM_SAMPLES

# FID stats file: auto-build if missing and FID is requested
if COMPUTE_FID and not os.path.exists('run/fid_stats_cifar10.npz'):
    os.makedirs('run', exist_ok=True)
    run('python fid_score.py --save-stats data/cifar10_images '
        'run/fid_stats_cifar10.npz --device cuda:0 --batch-size 256')

# original-model reference for SSIM -- SAME sample count as the pruned runs, so the
# paired SSIM comparison aligns batch-for-batch (mismatched counts crash compute_ssim)
run(f'python ddpm_sample.py --output_dir run/sample/ddpm_cifar10_pretrained '
    f'--batch_size {SAMPLE_BS} --total_samples {n_samples} --ddim_steps {DDIM_STEPS} '
    f'--model_path pretrained/ddpm_ema_cifar10 --skip_type uniform')
results = {}
for name, prune_cmd in METHODS.items():
    print('\n' + '='*60 + f'\n{name}\n' + '='*60, flush=True)
    run('rm -rf run/pruned/ddpm_cifar10_pruned '
        'run/finetuned/ddpm_cifar10_pruned_post_training run/sample/ddpm_cifar10_pruned')
    plog = cap(prune_cmd)             # prune (quiet while it runs; tail printed after)
    run(f'bash scripts/finetune_ddpm_cifar10.sh {ITERS} {WORKERS}')
    run(f'python ddpm_sample.py --output_dir run/sample/ddpm_cifar10_pruned '
        f'--batch_size {SAMPLE_BS} --total_samples {n_samples} --ddim_steps {DDIM_STEPS} '
        f'--pruned_model_ckpt run/finetuned/ddpm_cifar10_pruned_post_training/pruned/unet_ema_pruned.pth '
        f'--model_path run/finetuned/ddpm_cifar10_pruned_post_training --skip_type uniform')
    results[name] = {
        'params'  : grab(r'#Params:.*=>\s*([\d.]+)\s*M', plog),
        'macs'    : grab(r'#MACS:.*=>\s*([\d.]+)\s*G', plog),
        'jaccard' : grab(r'Jaccard\s*=\s*([\d.]+)', plog),
        'spearman': grab(r'Spearman rank corr\s*=\s*([\-\d.]+)', plog),
        'ssim'    : ssim_score(),
        'fid'     : fid_score() if COMPUTE_FID else float('nan'),
    }
    r = results[name]
    print(f"\n>>> {name}: params={r['params']:.2f}M  SSIM={r['ssim']:.4f}"
          + (f"  FID={r['fid']:.2f}" if COMPUTE_FID else ""))


## Results

In [ ]:
print(f'ratio={RATIO}  finetune={ITERS}  ssim_samples={SSIM_SAMPLES}  '
      f'fid={"on" if COMPUTE_FID else "off"}')
print('SSIM higher = better (closer to original) | Jacc/Spear vs magnitude (low = different cuts)\n')
cols = ['params', 'macs', 'ssim', 'jaccard', 'spearman'] + (['fid'] if COMPUTE_FID else [])
print(f"{'method':13s}" + ''.join(f'{c:>9s}' for c in cols))
for name, r in results.items():
    print(f"{name:13s}" + ''.join(f'{r[c]:9.3f}' for c in cols))
